# Cuaderno Científico de Cálculo II: Integración Simbólica, Cuadratura Numérica y Transformada de Fourier en DSP
**Curso:** Cálculo II (Cálculo Integral) — Ingeniería en Sistemas  
**Motor Matemático:** Python 3 (`SymPy` + `SciPy` + `NumPy`)

---
Este cuaderno ejecuta de forma aislada y auditable las **4 aplicaciones fundamentales del Cálculo Integral** empleadas en la plataforma de Cancelación Quirúrgica de Ruido:
1. **Resolución Simbólica Exacta (`SymPy`)**: Antiderivadas trigonométricas, Integración por Partes Cíclica ($\int e^{-\alpha t}\cos(\omega t)\,dt$) e Integrales Impropias de Tipo I ($\int_0^\infty$).
2. **Cuadratura Numérica Comparada (`SciPy`)**: Suma de Riemann (Izquierda y Punto Medio) vs. Regla del Trapecio (`scipy.integrate.trapezoid`) vs. Regla de Simpson 1/3 (`scipy.integrate.simpson`) vs. Valor Exacto por el Segundo Teorema Fundamental del Cálculo.
3. **Aplicaciones Físicas de la Integral Definida**: Energía Acústica Temporal $E = \int_0^T [f(t)]^2\,dt$, Valor Eficaz ($V_{\text{RMS}}$ por el Teorema del Valor Medio para Integrales) y verificación empírica de la **Identidad de Parseval**.
4. **Síntesis y Limpieza Quirúrgica de Audio `.wav`** mediante filtrado convolucional (`scipy.signal.iirnotch`).

In [ ]:
import sympy as sp
import numpy as np
from scipy import integrate, signal
from calculus_engine import (
    solve_symbolic_calculus,
    compare_quadrature_methods,
    process_audio_and_verify_parseval,
    synthesize_and_clean_wav_bytes,
)

sp.init_printing(use_unicode=True)
print(f"SymPy: {sp.__version__} | SciPy: {integrate.__name__} | NumPy: {np.__version__}")

## 1. Integración Simbólica con `SymPy` (Por Partes Cíclica e Integral Impropia)
Evaluamos simbólicamente la integral de proyección de una onda acústica amortiguada:
$$I = \int e^{-\alpha t} \cos(\omega t)\,dt \quad \text{y su límite impropio} \quad \lim_{T \to \infty} \int_0^T e^{-\alpha t} \cos(\omega t)\,dt$$

In [ ]:
t = sp.Symbol('t', real=True, positive=True)
T = sp.Symbol('T', real=True, positive=True)
alpha = sp.Symbol('alpha', real=True, positive=True)
omega = sp.Symbol('omega', real=True, positive=True)

integrando = sp.exp(-alpha * t) * sp.cos(omega * t)
antiderivada = sp.simplify(sp.integrate(integrando, t))
integral_definida = sp.simplify(sp.integrate(integrando, (t, 0, T)))
integral_impropia = sp.simplify(sp.integrate(integrando, (t, 0, sp.oo)))

print('1) Antiderivada exacta F(t) por Integración por Partes Cíclica:')
display(antiderivada)

print('\n2) Integral Definida en [0, T] (Segundo Teorema Fundamental del Cálculo F(T) - F(0)):')
display(integral_definida)

print('\n3) Integral Impropia de Tipo I en [0, ∞):')
display(integral_impropia)

## 2. Comparativa de Métodos de Integración Numérica (`Riemann` vs. `Trapecio` vs. `Simpson 1/3`)
Comparamos el error numérico al calcular la integral de Fourier $\int_0^T f(t)\cos(2\pi\xi t)\,dt$ en la frecuencia de resonancia del ruido ($\xi = 60\text{ Hz}$) frente a una frecuencia ortogonal ($\xi = 350\text{ Hz}$).

In [ ]:
res_60 = compare_quadrature_methods(probe_freq=60.0, noise_freq=60.0, partitions_n=200)
print(f"=== Sintonía en ξ = 60 Hz | Valor Exacto Analítico |f̂(60)| = {res_60['exact']['magnitude']:.6f} ===")
for key, m in res_60['methods'].items():
    print(f"{m['name']:46s} | Mag = {m['magnitude']:.6f} | Error Rel = {m['rel_error_pct']:.4f}% ({m['order']})")

res_350 = compare_quadrature_methods(probe_freq=350.0, noise_freq=60.0, partitions_n=200)
print(f"\n=== Ortogonalidad en ξ = 350 Hz | Valor Exacto Analítico |f̂(350)| = {res_350['exact']['magnitude']:.6f} ≈ 0 ===")

## 3. Energía Acústica, Valor Eficaz ($V_{\text{RMS}}$) y Verificación Empírica del Teorema de Parseval
Comprobamos que $\int_0^T |f(t)|^2\,dt = \int_{-\infty}^{\infty} |\hat{f}(\xi)|^2\,d\xi$ y medimos la reducción de energía tras aplicar el filtro Notch.

In [ ]:
fs = 44100
t_vec = np.arange(int(0.5 * fs)) / fs
senal_contaminada = (
    0.35 * np.sin(2 * np.pi * 140 * t_vec)
    + 0.25 * np.sin(2 * np.pi * 700 * t_vec)
    + 0.50 * np.sin(2 * np.pi * 60 * t_vec)
).tolist()

metricas = process_audio_and_verify_parseval(senal_contaminada, sample_rate=fs, notch_freq=60.0, q_factor=12.0)
print('Integrales en el Dominio del Tiempo (Simpson 1/3):', metricas['time_domain_integrals'])
print('Integrales en el Dominio de la Frecuencia (Parseval):', metricas['frequency_domain_integrals'])